# Workshop — 5. Orchestrate an Agentic Robot Loop

The fine-tuned SmolVLA remains the robot policy: it converts images, robot state, and language into joint actions. This notebook adds a configurable Strands Agent above that policy. The supervisor can use Ollama on the robot's companion computer or Amazon Bedrock when cloud access is available.

```text
user goal
   ↓
Strands Agent + Ollama or Bedrock (understand and plan)
   ↓ tool call
strands-robots run_policy(...)
   ↓ 30 Hz control loop
SmolVLA policy → joint actions → MuJoCo
   ↑                         ↓
   └──── images + state ─────┘
```

The user goal is free-form. The local LLM inspects the workspace, selects a supported robot skill and its arguments, observes the outcome, and decides whether to retry or ask for help. All components are defined visibly below.

## 1. Install and Configure the Runtime

SmolVLA always runs locally and provides motor behavior. For task-level reasoning, `AGENT_MODEL_PROVIDER=OLLAMA` keeps the complete inference loop on the robot's companion computer, while `BEDROCK` provides an optional connected mode. Here, **on-device** means the companion computer or edge compute module, not the motor-control microcontroller.

For the default Ollama path, start the server and download the model before running the agent cells. Skip these commands when selecting Bedrock:

```bash
ollama serve
ollama pull qwen3:4b
```

Set `STRANDS_LOCAL_MODEL` or `OLLAMA_HOST` to use another tool-capable local model or another computer on the robot's private network.

In [ ]:
%pip install -q -r requirements.txt

### Configure local rendering and inference

These settings select the correct MuJoCo renderer, enable PyTorch fallbacks on Apple Silicon, and expose Homebrew FFmpeg to the notebook process. They must be set before importing the robotics runtime.

In [ ]:
import os
import platform

os.environ.setdefault("MUJOCO_GL", "cgl" if platform.system() == "Darwin" else "egl")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("STRANDS_TRUST_REMOTE_CODE", "1")
if platform.system() == "Darwin":
    os.environ.setdefault("DYLD_FALLBACK_LIBRARY_PATH", "/opt/homebrew/lib")

os.environ["AGENT_MODEL_PROVIDER"] = "BEDROCK"
os.environ["STRANDS_BEDROCK_MODEL_ID"] = "global.openai.gpt-6-sol"
os.environ["STRANDS_LOCAL_MODEL"] = "qwen3.5:2b"

## 2. Locate the Fine-Tuned Checkpoint

Notebook 4 downloaded the SageMaker artifact under `outputs/smolvla-so100/evaluations/<training-job>/model`. We select the most recently modified complete local checkpoint rather than downloading it again.

In [ ]:
from pathlib import Path

MODEL_SEARCH_ROOT = Path("outputs/smolvla-so100/evaluations").resolve()
model_candidates = sorted(
    (config_path.parent for config_path in MODEL_SEARCH_ROOT.glob("*/model/config.json")),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)
if not model_candidates:
    raise FileNotFoundError(
        f"No downloaded fine-tuned checkpoint found under {MODEL_SEARCH_ROOT}. "
        "Run Notebook 4 first."
    )

LOCAL_MODEL_DIR = model_candidates[0]
print("Fine-tuned checkpoint:", LOCAL_MODEL_DIR)

## 3. Define and Build the Simulation

The agent needs the same experiment used for evaluation. We define the robot contract, reconstruct the target tray, add the two cameras, restore the dataset-aligned start pose, and implement the deterministic task metric.

In [ ]:
import mujoco
import numpy as np
import torch
from strands_robots import Robot
from strands_robots.policies import create_policy

INSTRUCTION = "Pick up the cube and place it in the box."
JOINT_KEYS = ["Rotation", "Pitch", "Elbow", "Wrist_Pitch", "Wrist_Roll", "Jaw"]
DATASET_MEAN_STATE = np.array(
    [14.4717, -55.7695, 54.3857, 63.2262, 85.8417, 9.3545],
    dtype=np.float64,
)
GRIPPER_JOINT_RANGE = (-0.175, 1.745)
BOX_CENTER = np.array([0.16, -0.30], dtype=np.float64)
FPS = 30


def require_success(result, operation):
    """Raise when a strands-robots operation returns an error envelope."""
    if result.get("status") == "success":
        return result
    text = " | ".join(
        str(item.get("text"))
        for item in result.get("content", [])
        if isinstance(item, dict) and item.get("text")
    )
    raise RuntimeError(f"{operation} failed: {text or result}")


def dataset_mean_action():
    """Convert the original real-data mean pose to MuJoCo units."""
    values = np.empty(6, dtype=np.float64)
    values[:5] = np.deg2rad(DATASET_MEAN_STATE[:5])
    jaw_min, jaw_max = GRIPPER_JOINT_RANGE
    values[5] = jaw_min + (DATASET_MEAN_STATE[5] / 100.0) * (jaw_max - jaw_min)
    return dict(zip(JOINT_KEYS, values.tolist(), strict=True))

### Instantiate the physical experiment

The scene builder makes the experiment repeatable: it adds the same cube, tray, cameras, and start pose used during evaluation. The diagnostic function remains separate so task success can be checked without asking the LLM.

In [ ]:
def build_scene():
    """Create one fresh SO100 pick-place scene."""
    sim = Robot("so100", mesh=False)
    require_success(
        sim.add_object(
            name="cube",
            shape="box",
            position=[0.0, -0.35, 0.015],
            size=[0.03, 0.03, 0.03],
            color=[0.9, 0.12, 0.08, 1.0],
            mass=0.03,
        ),
        "add cube",
    )
    blue = [0.12, 0.28, 0.75, 1.0]
    x, y = BOX_CENTER
    parts = {
        "target_base": ([x, y, 0.005], [0.12, 0.12, 0.01]),
        "target_left": ([x - 0.055, y, 0.025], [0.01, 0.12, 0.05]),
        "target_right": ([x + 0.055, y, 0.025], [0.01, 0.12, 0.05]),
        "target_back": ([x, y + 0.055, 0.025], [0.10, 0.01, 0.05]),
        "target_front": ([x, y - 0.055, 0.025], [0.10, 0.01, 0.05]),
    }
    for name, (position, size) in parts.items():
        require_success(
            sim.add_object(
                name=name,
                shape="box",
                position=list(position),
                size=list(size),
                color=blue,
                is_static=True,
            ),
            f"add {name}",
        )
    require_success(
        sim.add_camera(
            name="top",
            position=[0.45, -0.60, 0.42],
            target=[0.06, -0.31, 0.06],
            fov=55,
            width=640,
            height=480,
        ),
        "add top camera",
    )
    require_success(
        sim.add_camera(
            name="wrist",
            position=[0.02, -0.56, 0.18],
            target=[0.04, -0.31, 0.04],
            fov=70,
            width=640,
            height=480,
        ),
        "add wrist camera",
    )
    require_success(
        sim.send_action(dataset_mean_action(), robot_name="so100", n_substeps=600),
        "move SO100 to the dataset-mean pose",
    )
    return sim


def task_diagnostics(sim):
    """Return cube position and the authoritative placed-in-box metric."""
    cube_id = mujoco.mj_name2id(sim.mj_model, mujoco.mjtObj.mjOBJ_BODY, "cube")
    position = np.asarray(sim.mj_data.xpos[cube_id], dtype=float).copy()
    inside_box_xy = bool(
        np.all(np.abs(position[:2] - BOX_CENTER) < np.array([0.045, 0.045]))
    )
    return {
        "cube_position_m": np.round(position, 4).tolist(),
        "placed_in_box": inside_box_xy and position[2] < 0.10,
    }


sim = build_scene()
print("Initial task state:", task_diagnostics(sim))

## 4. Inspect the Native `strands-robots` Tool

`Robot("so100")` is itself a stateful Strands `AgentTool`. The agent sees one JSON tool with an `action` field. Its schema includes scene operations, sensors, recording, and `run_policy`. A general robot agent could receive this full tool directly.

In [ ]:
native_schema = sim.tool_spec["inputSchema"]["json"]
native_actions = native_schema["properties"]["action"]["enum"]

print("Agent tool name:", sim.tool_name)
print("Published actions:", len(native_actions))
print("Contains run_policy:", "run_policy" in native_actions)
print("policy_config schema:", native_schema["properties"]["policy_config"])
print("stop_when schema:", native_schema["properties"]["stop_when"])

## 5. Load the Fine-Tuned Policy

The checkpoint was trained on simulation-native camera names and radians. We construct the embodiment adapter explicitly and load the model once. Every agent-requested segment will reuse this same in-memory policy object.

In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
finetuned_embodiment = {
    "name": "so100_smolvla_sim_finetuned",
    "obs_rename": {
        "wrist": "observation.images.wrist",
        "top": "observation.images.top",
    },
    "state_keys": JOINT_KEYS,
    "action_keys": JOINT_KEYS,
    "dim_policy": "strict",
    "state_units": "radians",
    "action_units": "radians",
}

finetuned_policy = create_policy(
    "lerobot_local",
    pretrained_name_or_path=str(LOCAL_MODEL_DIR),
    policy_type="smolvla",
    device=device,
    embodiment=finetuned_embodiment,
    strict_keys=True,
)

print("Policy adapter:", type(finetuned_policy).__name__)
print("Device:", device)
print("Execution horizon:", finetuned_policy.execution_horizon)

## 6. Define the Agent's State and Full Task Horizon

The LLM must not own safety or resource limits. One 400-step segment lasts about 13.3 seconds at 30 Hz, matching the complete rollouts used in Notebooks 2 and 4. Python permits at most two segments (800 steps total). We deliberately run the complete segment: entering the box region is not yet task completion because the policy still needs time to release the cube and retreat.

In [ ]:
import json
from dataclasses import dataclass, field

SEGMENT_STEPS = 400
MAX_TOTAL_STEPS = 800
VIDEO_DIR = Path("outputs/agentic-pick").resolve()


@dataclass
class AgenticLoopState:
    """Track the hard policy budget and every completed segment."""
    segment_steps: int
    max_total_steps: int
    total_steps_used: int = 0
    attempts: list[dict] = field(default_factory=list)
    escalations: list[dict] = field(default_factory=list)
    terminal_error: str | None = None
    escalated: bool = False

    @property
    def remaining_steps(self):
        return max(0, self.max_total_steps - self.total_steps_used)


loop_state = AgenticLoopState(SEGMENT_STEPS, MAX_TOTAL_STEPS)
print("Total policy budget:", loop_state.max_total_steps)

## 7. Build Three Least-Privilege Agent Tools

Instead of exposing all simulation actions, this experiment gives the agent only:

- `inspect_robot_workspace()`: discover entities, supported skills, state, and budget;
- `execute_robot_skill(...)`: translate structured skill arguments into a canonical VLA instruction and execute it;
- `request_human_help(...)`: stop physical execution and record an escalation.

The LLM chooses the tool and arguments from the free-form user goal. Python validates that choice against the robot's capability catalog before any physical action.

In [ ]:
from strands import tool


def result_json(result):
    """Extract the first JSON block from a strands-robots result."""
    return next(
        (
            item["json"]
            for item in result.get("content", [])
            if isinstance(item, dict) and isinstance(item.get("json"), dict)
        ),
        {},
    )


def result_text(result):
    """Join human-readable text blocks from a tool result."""
    return " | ".join(
        str(item["text"])
        for item in result.get("content", [])
        if isinstance(item, dict) and item.get("text")
    )


def tool_envelope(payload, status="success"):
    """Return the standard Strands status/content envelope."""
    return {
        "status": status,
        "content": [
            {"text": json.dumps(payload, sort_keys=True)},
            {"json": payload},
        ],
    }


SUPPORTED_SKILLS = {
    "pick_and_place": {
        "targets": ["cube"],
        "destinations": ["box"],
        "instruction_template": "Pick up the {target} and place it in the {destination}.",
    }
}


@tool
def inspect_robot_workspace() -> dict:
    """Discover robot capabilities and authoritative task state without moving."""
    diagnostics = task_diagnostics(sim)
    return tool_envelope({
        "task_success": bool(diagnostics["placed_in_box"]),
        "diagnostics": diagnostics,
        "available_entities": {"targets": ["cube"], "destinations": ["box"]},
        "supported_skills": SUPPORTED_SKILLS,
        "segments_completed": len(loop_state.attempts),
        "escalated": loop_state.escalated,
        "steps_used": loop_state.total_steps_used,
        "remaining_steps": loop_state.remaining_steps,
    })

### Wrap the real policy rollout

This tool is the action boundary. The local LLM supplies a skill, target, and destination; Python validates them and constructs the canonical instruction seen during policy training. Unsupported requests are refused before the robot moves.

In [ ]:
@tool
def execute_robot_skill(skill: str, target: str, destination: str) -> dict:
    """Execute one supported robot skill through the local SmolVLA policy.

    Args:
        skill: Skill name discovered through inspect_robot_workspace.
        target: Object to manipulate, using its workspace name.
        destination: Named destination for the object.
    """
    contract = SUPPORTED_SKILLS.get(skill)
    if contract is None:
        return tool_envelope({
            "error": f"Unsupported skill: {skill!r}",
            "supported_skills": sorted(SUPPORTED_SKILLS),
        }, status="error")
    if target not in contract["targets"] or destination not in contract["destinations"]:
        return tool_envelope({
            "error": "The requested target or destination is unsupported.",
            "supported_targets": contract["targets"],
            "supported_destinations": contract["destinations"],
        }, status="error")
    if loop_state.escalated:
        return tool_envelope({
            "error": "Physical execution is disabled after escalation.",
            "task_success": False,
        }, status="error")

    policy_instruction = contract["instruction_template"].format(
        target=target,
        destination=destination,
    )
    diagnostics = task_diagnostics(sim)
    if diagnostics["placed_in_box"]:
        return tool_envelope({
            "run_status": "skipped",
            "task_success": True,
            "steps_used": 0,
            "remaining_steps": loop_state.remaining_steps,
            "diagnostics": diagnostics,
        })
    if loop_state.terminal_error is not None:
        return tool_envelope({
            "error": "A previous rollout ended with a software error.",
            "detail": loop_state.terminal_error,
            "task_success": False,
            "remaining_steps": loop_state.remaining_steps,
        }, status="error")
    if loop_state.remaining_steps == 0:
        return tool_envelope({
            "error": "The policy-step budget is exhausted.",
            "task_success": False,
            "steps_used": loop_state.total_steps_used,
            "remaining_steps": 0,
        }, status="error")

    steps = min(loop_state.segment_steps, loop_state.remaining_steps)
    segment_number = len(loop_state.attempts) + 1
    VIDEO_DIR.mkdir(parents=True, exist_ok=True)
    video_path = VIDEO_DIR / f"segment_{segment_number:02d}.mp4"

    result = sim.run_policy(
        robot_name="so100",
        policy_object=finetuned_policy,
        instruction=policy_instruction,
        n_steps=steps,
        control_frequency=FPS,
        fast_mode=True,
        video={"path": str(video_path), "camera": "top", "fps": FPS},
    )
    report = result_json(result)
    measured_steps = int(report.get("steps_used", report.get("n_steps", 0)) or 0)
    loop_state.total_steps_used += min(measured_steps, loop_state.remaining_steps)
    diagnostics = task_diagnostics(sim)

    attempt = {
        "segment": segment_number,
        "skill": skill,
        "target": target,
        "destination": destination,
        "policy_instruction": policy_instruction,
        "run_status": result.get("status"),
        "task_success": bool(diagnostics["placed_in_box"]),
        "steps_requested": steps,
        "steps_used": measured_steps,
        "remaining_steps": loop_state.remaining_steps,
        "stopped_reason": report.get("stopped_reason"),
        "action_errors": report.get("action_errors"),
        "partial_action_failure_rate": report.get("partial_action_failure_rate"),
        "diagnostics": diagnostics,
        "video_path": str(video_path),
        "error_text": result_text(result) if result.get("status") != "success" else None,
    }
    loop_state.attempts.append(attempt)
    if result.get("status") != "success":
        loop_state.terminal_error = attempt["error_text"] or "Unknown run_policy error."
    return tool_envelope(
        attempt,
        status="success" if result.get("status") == "success" else "error",
    )


@tool
def request_human_help(reason: str) -> dict:
    """Stop further physical execution and record why assistance is required.

    Args:
        reason: Concise explanation of the unsupported goal or repeated failure.
    """
    escalation = {
        "reason": reason,
        "diagnostics": task_diagnostics(sim),
        "steps_used": loop_state.total_steps_used,
    }
    loop_state.escalations.append(escalation)
    loop_state.escalated = True
    return tool_envelope({"escalated": True, **escalation})


agent_tools = [inspect_robot_workspace, execute_robot_skill, request_human_help]
for agent_tool in agent_tools:
    print(agent_tool.tool_name, agent_tool.tool_spec["inputSchema"])

## 8. Write the Supervisor Contract

The system prompt defines the agent's task-level responsibilities. The hard budget remains enforced by Python even if the model ignores these instructions.

In [ ]:
SYSTEM_PROMPT = f"""You are the local high-level supervisor running on a robot companion computer.

Interpret the user's free-form goal, but never command robot joints yourself.
The local SmolVLA policy owns joint actions. You may only use these tools:
1. inspect_robot_workspace: discover entities, supported skills, physical state, and remaining budget;
2. execute_robot_skill: choose a supported skill, target, and destination;
3. request_human_help: stop physical execution when the goal is unsupported or recovery is exhausted.

Follow this loop:
- Always inspect before acting.
- Map the user goal onto exactly one skill and entity pair returned by inspection.
- Never invent a skill, target, or destination.
- Execute one segment, then inspect the deterministic task result.
- If the task is incomplete and budget remains, retry at most once.
- If the goal is unsupported, a tool fails, or the retry does not succeed, request human help.

The hard policy budget is {MAX_TOTAL_STEPS} steps and is enforced by Python.
run_status='success' means only that software executed. Report physical success
only when task_success=true. Keep the final answer concise and state the skill,
target, destination, and deterministic verdict."""

print(SYSTEM_PROMPT)

## 9. Select the Agent Model Provider

`create_agent_model(provider)` isolates model-provider configuration from the robot loop. Choose `OLLAMA` for on-device reasoning or `BEDROCK` for a connected deployment; tools, system prompt, policy, and safety limits remain unchanged.

- Ollama reads `OLLAMA_HOST` and `STRANDS_LOCAL_MODEL`.
- Bedrock reads `AWS_REGION` and `STRANDS_BEDROCK_MODEL_ID`.

In [ ]:
from urllib.error import URLError
from urllib.request import urlopen

from strands import Agent


def create_agent_model(provider: str):
    """Create the task-level reasoning model for OLLAMA or BEDROCK.

    Args:
        provider: Model provider name: `OLLAMA` or `BEDROCK`.
    """
    normalized = provider.strip().upper()

    if normalized == "OLLAMA":
        from strands.models.ollama import OllamaModel

        host = os.environ.get("OLLAMA_HOST", "http://127.0.0.1:11434").rstrip("/")
        if "://" not in host:
            host = "http://" + host
        model_id = os.environ.get("STRANDS_LOCAL_MODEL", "qwen3:4b")

        try:
            with urlopen(f"{host}/api/tags", timeout=3) as response:
                tags = json.load(response)
        except (URLError, TimeoutError) as error:
            raise RuntimeError(
                f"Cannot reach Ollama at {host}. Start it with: ollama serve"
            ) from error

        available = {
            model_name
            for item in tags.get("models", [])
            if (model_name := item.get("name") or item.get("model"))
        }
        if model_id not in available:
            raise RuntimeError(
                f"Local model {model_id!r} is not installed. "
                f"Run: ollama pull {model_id}. Available: {sorted(available)}"
            )
        return OllamaModel(
            host=host,
            model_id=model_id,
            temperature=0.0,
            keep_alive="15m",
        )

    if normalized == "BEDROCK":
        import boto3
        from strands.models import BedrockModel

        region = os.environ.get("AWS_REGION") or boto3.Session().region_name or "us-east-1"
        model_id = os.environ.get(
            "STRANDS_BEDROCK_MODEL_ID",
            "global.anthropic.claude-sonnet-4-6",
        )
        return BedrockModel(model_id=model_id, region_name=region)

    raise ValueError(
        f"Unsupported agent model provider {provider!r}. Choose 'OLLAMA' or 'BEDROCK'."
    )


AGENT_MODEL_PROVIDER = os.environ.get("AGENT_MODEL_PROVIDER", "OLLAMA").upper()
agent_model = create_agent_model(AGENT_MODEL_PROVIDER)
agent = Agent(
    model=agent_model,
    tools=agent_tools,
    system_prompt=SYSTEM_PROMPT,
    callback_handler=None,
)

print("Agent model provider:", AGENT_MODEL_PROVIDER)
print("Agent model config:", agent_model.get_config())

### Provide a free-form user goal

The wording deliberately differs from SmolVLA's canonical training instruction. The local LLM must inspect the workspace, map `red block` and `blue tray` onto the available entities, select the supported skill, and supply its structured arguments.

In [ ]:
USER_GOAL = os.environ.get(
    "ROBOT_USER_GOAL",
    "Please tidy the workspace by putting the red block into the blue tray.",
)

print("User goal:", USER_GOAL)
agent_result = agent(USER_GOAL)
print(agent_result)

## 10. Audit the Outcome

Agent prose is not the experiment record. We inspect the Python-owned attempt history and recompute the deterministic physical metric.

In [ ]:
import pandas as pd
from IPython.display import Video, display

display(pd.DataFrame(loop_state.attempts))
if loop_state.escalations:
    display(pd.DataFrame(loop_state.escalations))
print("Final deterministic diagnostics:", task_diagnostics(sim))
print("Total policy steps used:", loop_state.total_steps_used)

for attempt in loop_state.attempts:
    video_path = Path(attempt["video_path"])
    if video_path.is_file():
        print(f"Segment {attempt['segment']}: {video_path}")
        display(Video(str(video_path), embed=True, width=640))

## What the Agent Adds — and What It Does Not

The local agent now performs genuine task-level orchestration: it interprets a free-form goal, discovers capabilities, selects a tool and structured arguments, observes deterministic feedback, retries within a hard budget, and escalates when necessary. SmolVLA remains the only component that emits joint actions.

With Ollama selected, the complete inference loop can run on a robot companion computer. Bedrock provides the same agent interface for connected environments. Neither path performs online weight updates: dataset collection, fine-tuning, evaluation, and checkpoint promotion remain a separate learning lifecycle. Production hardware also requires independent safety interlocks, authorization, timeouts, and emergency stop.